# Offline BTCUSDT training on Snowflake
This notebook performs no network calls. Upload the repository (including `src/` and `data/`) to the notebook workspace or mount a Snowflake stage first. Select a GPU runtime if available.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
DATA = ROOT / 'data' / 'btcusdt_1s'
if not DATA.exists():
    DATA = ROOT / 'data' / 'sample' / 'btcusdt_1s_sample.csv'
print('Data:', DATA)

In [ ]:
from bitcoin_predictor.train import run
metrics = run(str(DATA), str(ROOT / 'artifacts'), speed=0.0)
metrics

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, clear_output
files = sorted((ROOT/'artifacts'/'predictions').glob('epoch_*.csv'))
frame = pd.concat([pd.read_csv(f) for f in files[-4:]], ignore_index=True).tail(7200)
fig = go.Figure()
fig.add_scatter(x=frame.timestamp, y=frame.actual_price, name='Actual')
fig.add_scatter(x=frame.timestamp, y=frame.predicted_price_30m, name='Predicted t+30m')
fig.show()
metrics_path = ROOT/'artifacts'/'latest_metrics.json'
print(json.loads(metrics_path.read_text()) if metrics_path.exists() else {})

## Live display
Run training in bounded blocks and re-run the plot cell to refresh. Snowflake cells are not browser-side streaming processes; checkpoint files make updates durable and reproducible.